#  [Mini-Project 1: London Air Quality Analysis]: NB2 - Transforming Collected Data

In [18]:
import os
import json
import requests
import numpy as np
import pandas as pd
import csv

from dotenv import load_dotenv

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Section 1: JSON loaded and parsed

**Load Pollution Data from JSON File and Verify Structure**

In [19]:
with open("data/london_pollution__2022_2026.json", "r") as f:
    pollution_data = json.load(f)

print(type(pollution_data))
print('Data loaded and parsed.')
print(f"We have all {len(pollution_data)} locations")

<class 'list'>
Data loaded and parsed.
We have all 2 locations


<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: NA
2. My understanding: I think here I'm opening the JSON file I just saved in r - read - mode. Then with json.load(f), f being the opened json, I am converting it into python (to be a list of dictionaries). Then, I'm just checking the type, and printing confirmation of loading and parsing (converting) everything. 

</div>

## Section 2: Pandas transformations applied

**Flatten Nested JSON Structure and Add Location Names to Each Reading**

In [20]:
all_readings = []

for location in pollution_data:
    readings = location["list"]
    location_name = location.get("location")  # Get the location name
    for reading in readings:
        reading['location'] = location_name  # Add it to each reading
    all_readings.extend(readings)

<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: NA
2. My understanding: So I'm looping through each location. Within the loop I'm extracting the readings (which are hourly), for each location, and getting the location name into a variable with .get.    
Honestly, I'm still kinda confused on why I need to loop in readings to add the location again. Claude mentioned that the og loaction object with the name gets discarded after the loop iteration ends, and with "reading['location'] = location_name" I'm adding the location as a key (not a extra value in list), which labels all the values with the location.
3. For Loops: I need for loops for a similar reason as the last time, as the JSON structure is nested unlike a df, so I can't vectorize it.

</div>

**Convert all_readings into DataFrame and Extract Nested Components**

In [21]:
df = pd.DataFrame(all_readings)

print(df.head())
print(df.shape)
print(df.columns) #Shows the basic tabular structure

#-----------------------------------------------------------------

df['aqi'] = df['main'].apply(lambda x: x.get('aqi'))

df['co'] = df['components'].apply(lambda x: x.get('co'))
df['no'] = df['components'].apply(lambda x: x.get('no'))
df['no2'] = df['components'].apply(lambda x: x.get('no2'))
df['o3'] = df['components'].apply(lambda x: x.get('o3'))
df['pm2_5'] = df['components'].apply(lambda x: x.get('pm2_5'))
df['pm10'] = df['components'].apply(lambda x: x.get('pm10'))
df['so2'] = df['components'].apply(lambda x: x.get('so2'))
df['nh3'] = df['components'].apply(lambda x: x.get('nh3'))

df = df.drop(columns=['main', 'components'])
print(df.head()) #Prints the first 5 rows by default
print(df.columns)


         main                                         components          dt  \
0  {'aqi': 1}  {'co': 257.02, 'no': 0, 'no2': 6.34, 'o3': 72....  1648767600   
1  {'aqi': 1}  {'co': 257.02, 'no': 0, 'no2': 6.34, 'o3': 75....  1648771200   
2  {'aqi': 1}  {'co': 257.02, 'no': 0, 'no2': 6.25, 'o3': 78....  1648774800   
3  {'aqi': 2}  {'co': 260.35, 'no': 0, 'no2': 5.83, 'o3': 80....  1648778400   
4  {'aqi': 2}  {'co': 263.69, 'no': 0, 'no2': 5.57, 'o3': 80....  1648782000   

                              location  
0  Equestrian_Statue_of_King_Charles_I  
1  Equestrian_Statue_of_King_Charles_I  
2  Equestrian_Statue_of_King_Charles_I  
3  Equestrian_Statue_of_King_Charles_I  
4  Equestrian_Statue_of_King_Charles_I  
(66194, 4)
Index(['main', 'components', 'dt', 'location'], dtype='str')
           dt                             location  aqi      co   no   no2  \
0  1648767600  Equestrian_Statue_of_King_Charles_I  1.0  257.02  0.0  6.34   
1  1648771200  Equestrian_Statue_of_King_Char

<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: **.get()** on python dictionaries in pandas (if I got that right). I was getting KeyError without it, so claude helped me learn that lambda x: x.get() safely extracts the value in the get command like normal get, and I can apply it to each pandas cell.         
**Using lambda with .apply() on nested dictionaries** was something I did not know how to do. I of course knew lambda with simple equations like (lambda x: x > 15) from W05, but claude helped me learn how to use it for multiple values in nested dictionaries with x: x.get           
**.drop** - I was having trouble because after extracting the chemicals and such from the nested dictionaries they didn't leave but duplicated, so claude taught me how to remove the columns with nested dictionaries via df.drop 
2. My understanding: OK, so I'm first converting all_readings into a DataFrame (Table) with "df = pd.DataFrame(all_readings)" Then I'm just inspecting the structure to help with the following code.     
After this, I'm extracting each chemical/aqi/particles from the nested 'main' or 'components' dictionary I saw via inspecting the structure.        
Now with my newly learned .drop, I delete the now redundant nested og columns

</div>

## Section 3: Day-of-week tagging and temporal processing

**Convert Unix Timestamps to Datetime and Extract Temporal Components**

In [23]:
df['dt'] = pd.to_datetime(df['dt'], unit='s', utc=True)
df['dayofweek'] = df['dt'].dt.dayofweek

df['is_weekend'] = df['dt'].dt.dayofweek >= 5

df['year'] = df['dt'].dt.year
df['month'] = df['dt'].dt.month
df['day'] = df['dt'].dt.day

day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['weekday_name'] = df['dayofweek'].apply(lambda x: day_names[x])

<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: **Lambda to index into a list**: Slides showed lambda for comparisons, I saw how I used lambda with .get and adapted it to day_names[x], so it looks up the day name for each number.
2. My understanding: **df['dt'] = pd.to_datetime(df['dt'], unit='s', utc=True)**: converts UNIX to readable datetime format, then I extract with dt.dayofweek,     
**df['is_weekend'] = df['dt'].dt.dayofweek >= 5**: Boolean: true if saturday/sunday (5,6) false if otherwise
Then I just extract the year, month, then day. Lastly, see Advanced Features

</div>

## Section 4: CSV file(s) saved to data/ folder

**Saving new DataFrame as CSV file**

In [24]:
df.to_csv("data/london_pollution_processed.csv", index=False)

print("Saved to data/london_pollution_processed.csv")

Saved to data/london_pollution_processed.csv


<div style="font-family: system-ui; padding: 10px 10px 10px 10px; background-color: #FFFFFF; border-left: 8px solid #000000ff; border-radius: 5px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); max-width: 600px; color: #212121; font-size: 14px;">

**Info + my understanding of cell above**
1. Advanced features: **Index=False**
2. My understanding: Index=False parameter prevents pandas from adding a row index column. I had it on default and saw the extra column when loading it in NB03, so asked claude how to remove it.

</div>